# LAB-14 · Fuse SiLU × multiply

Like a chef finishing a dish at one counter, fusion keeps an intermediate result close instead of carrying it to storage. You will combine SiLU (a smooth gate, a / (1 + exp(-a))) and multiplication into one kernel, then count and measure the memory trips.

~20 minutes · **GPU optional**. A kernel is a small function run over many data items; Triton is a Python-based language for writing it. Run cells in order, fill the `# TODO` gaps, then use the checks; starter failures are expected.

Use a fresh Python runtime. Setup can install Triton in your account; it needs no API key or repository files. CPU correctness uses Triton's interpreter on a supported Linux runtime (Colab/Kaggle); native macOS is not a supported Triton runtime.


In [ ]:
import os
import sys
import torch

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
# The interpreter setting must be chosen BEFORE the first import of Triton.
if "triton" in sys.modules:
    raise RuntimeError("Restart the runtime, then run this setup cell first (Triton is already imported).")
if HAS_GPU:
    os.environ.pop("TRITON_INTERPRET", None)
else:
    os.environ["TRITON_INTERPRET"] = "1"
    print("No GPU: correctness checks only, timings skipped")
try:
    import triton
except ImportError:
    # Runs only in your notebook runtime; PyTorch is preinstalled in Colab/Kaggle.
    %pip install -q triton
    import triton
import triton.language as tl

GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "none (CPU interpreter)"
print(f"torch {torch.__version__}; triton {triton.__version__}; GPU: {GPU_NAME}")
torch.manual_seed(0)


In [ ]:
# Self-check failures are reported without stopping Run all, so you can reach Solution.
# Solution checks are strict: a broken reference stops execution before benchmarking.
def check(label, test, strict=False):
    try:
        test()
    except Exception as error:
        print(f"FAIL {label}: {type(error).__name__}: {error}")
        if strict:
            raise
        return False
    print(f"PASS {label}")
    return True

# Vendor datasheet peaks, decimal GB/s. Match variants conservatively.
PEAK_GB_S = {"T4": 320.0, "P100 16 GB": 732.0, "H100 SXM": 3350.0}
def detected_peak():
    if not HAS_GPU:
        return None
    name = GPU_NAME.upper()
    if name in ("TESLA T4", "NVIDIA T4", "T4"):
        return PEAK_GB_S["T4"]
    memory = torch.cuda.get_device_properties(0).total_memory
    if "P100" in name and 15e9 <= memory <= 18e9:
        return PEAK_GB_S["P100 16 GB"]
    if "H100" in name and "SXM" in name:
        return PEAK_GB_S["H100 SXM"]
    return None

def report(label, milliseconds, byte_count):
    gb_s = byte_count / (milliseconds * 1e6)
    text = f"your run | {label}: {milliseconds:.4f} ms, {byte_count:,} logical bytes, {gb_s:.2f} GB/s"
    peak = detected_peak()
    if peak is not None:
        text += f", {100 * gb_s / peak:.1f}% of vendor peak"
    print(text)

def bench(label, call, byte_count):
    if not HAS_GPU:
        print(f"Timings skipped: {label} (CPU correctness only)")
        return
    from triton.testing import do_bench
    milliseconds = do_bench(call, warmup=25, rep=100)
    report(label, milliseconds, byte_count)


## 1 · Finish the gate at the counter

The separate PyTorch operations write the SiLU result to a temporary tensor before multiplication reads it back. Fill in the gated product below, using the loaded values inside one program. All examples use float32 (four bytes per number).


In [ ]:
@triton.jit
def silu_mul_kernel(A, B, Out, N: tl.constexpr, BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    a = tl.load(A + offsets, mask=mask, other=0)
    b = tl.load(B + offsets, mask=mask, other=0)
    value = a * b  # TODO: apply the SiLU gate to a before multiplying
    tl.store(Out + offsets, value, mask=mask)

def fused(a, b, out=None):
    assert a.shape == b.shape and a.is_contiguous() and b.is_contiguous()
    if out is None:
        out = torch.empty_like(a)
    if a.numel():
        silu_mul_kernel[(triton.cdiv(a.numel(), 256),)](a, b, out, a.numel(), 256)
    return out


In [ ]:
def check_fused():
    for n in (0, 1, 257, 1000):
        a = torch.linspace(-20, 20, n, device=DEVICE, dtype=torch.float32)
        b = torch.randn_like(a)
        expected = torch.nn.functional.silu(a) * b
        torch.testing.assert_close(fused(a, b), expected, rtol=1e-4, atol=1e-5)
check("fused SiLU times b", check_fused)


## 2 · Count the trips

Count each full tensor read or write, as if counting boxes through a warehouse door. Separate SiLU and multiply move five element-sized values per output; the fused kernel moves three. Complete the byte counter; this is logical accounting, not a hardware traffic measurement.


In [ ]:
def traffic(n, element_bytes):
    # Unfused: read a, write temporary; read temporary and b, write output.
    # Fused: read a and b, write output.
    return torch.tensor([0, 0  # TODO: unfused bytes, fused bytes
    ], dtype=torch.int64)


In [ ]:
def check_traffic():
    for n in (0, 1, 1000):
        expected = torch.tensor([20 * n, 12 * n], dtype=torch.int64)
        torch.testing.assert_close(traffic(n, 4), expected, rtol=0, atol=0)
check("logical bytes: unfused and fused FP32", check_traffic)


## Try first

Try filling the TODOs and rerunning each exercise and its check before reading on. Run all intentionally reports starter failures, then installs the complete definitions below and checks them again. Only checked solutions are timed.


## Solution


In [ ]:
@triton.jit
def silu_mul_kernel(A, B, Out, N: tl.constexpr, BLOCK: tl.constexpr):
    offsets = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    a = tl.load(A + offsets, mask=mask, other=0)
    b = tl.load(B + offsets, mask=mask, other=0)
    value = (a / (1.0 + tl.exp(-a))) * b
    tl.store(Out + offsets, value, mask=mask)

def fused(a, b, out=None):
    assert a.shape == b.shape and a.is_contiguous() and b.is_contiguous()
    if out is None:
        out = torch.empty_like(a)
    if a.numel():
        silu_mul_kernel[(triton.cdiv(a.numel(), 256),)](a, b, out, a.numel(), 256)
    return out


In [ ]:
def traffic(n, element_bytes):
    # Unfused: read a, write temporary; read temporary and b, write output.
    # Fused: read a and b, write output.
    return torch.tensor([5 * n * element_bytes, 3 * n * element_bytes], dtype=torch.int64)


In [ ]:
check("solution: fused values", check_fused, strict=True)
check("solution: byte accounting", check_traffic, strict=True)


## Measure on a GPU

Results below are **your run**, not a promised speedup. Byte counts are logical tensor reads/writes, excluding allocation, cache effects and instruction overhead; GB/s is effective bandwidth, not measured HBM traffic, and can exceed a datasheet peak when data is cached. Times depend on shape, GPU, compiler and runtime.

The small peak table uses vendor datasheets: T4 320 GB/s, P100 **16 GB** 732 GB/s, H100 **SXM** 3.35 TB/s. Ambiguous or unknown GPU variants show GB/s without a peak percentage. No timings run in CPU-interpreter mode.


In [ ]:
if HAS_GPU:
    n = 2**22  # Illustrative workload, not a benchmark claim.
    a = torch.randn(n, device=DEVICE)
    b = torch.randn_like(a)
    temporary = torch.empty_like(a)
    out = torch.empty_like(a)
    def unfused():
        torch.ops.aten.silu.out(a, out=temporary)
        torch.mul(temporary, b, out=out)
        return out
    torch.testing.assert_close(unfused(), torch.nn.functional.silu(a) * b, rtol=1e-4, atol=1e-5)
    unfused_bytes, fused_bytes = traffic(n, a.element_size()).tolist()
    print("Logical FP32 bytes/output: unfused 20, fused 12; ratio 5/3. This is not a speedup promise.")
    bench("two eager PyTorch kernels", unfused, unfused_bytes)
    bench("one fused Triton kernel", lambda: fused(a, b, out), fused_bytes)
else:
    print("Timings skipped (CPU correctness only)")


## Take it further

You built one gate-and-multiply kernel and counted the intermediate read/write it removes. Real systems can fuse more surrounding work, but register use and kernel launch costs affect the result. Backs I28 (kernel fusion).

[Academy LAB-14 page](https://tile-and-error-academy.vercel.app/#lab/lab-14) · [Tile & Error channel](https://www.youtube.com/@tileanderror)
